In [1]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../config"

import logging

logging.basicConfig(level="DEBUG")

# Sync example Tidal and Spotify

In this example we will show how to sync a playlist from Spotify to Tidal. This is a pretty easy use case, because tracks on both services can be identified by their [ISRC](https://en.wikipedia.org/wiki/International_Standard_Recording_Code).

To execute this example you will need to configure both the spotify and tidal services.


## The initial playlist

We start with a spotify playlist here and create a tidal playlist with the same name if it does not exist yet. You can
also start with two blank playlists.


In [2]:
from plistsync.services.spotify import (
    SpotifyLibraryCollection,
)

spotify_library = SpotifyLibraryCollection()

# Spotify id of the playlist to sync (Drum & Bass Top 100)
# By name
if spotify_playlist := spotify_library.get_playlist(
    url="https://open.spotify.com/playlist/0Zarq4BVkFkZOWkmqsfrjA"
):
    print(
        f"Found playlist: {spotify_playlist.name} "
        f"({len(spotify_playlist.tracks)} tracks)"
    )
else:
    print("Playlist not found.")

INFO:eyconf.config.file:Loading config file: /Users/paul/para/2_Projects/plistsync_25.nosync/config/config.yml
INFO:eyconf.config.file:Loading config file: /Users/paul/para/2_Projects/plistsync_25.nosync/config/config.yml
INFO:eyconf.config.file:Loading config file: /Users/paul/para/2_Projects/plistsync_25.nosync/config/config.yml
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): accounts.spotify.com:443
DEBUG:urllib3.connectionpool:https://accounts.spotify.com:443 "POST /api/token HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.spotify.com:443
DEBUG:urllib3.connectionpool:https://api.spotify.com:443 "GET /v1/playlists/0Zarq4BVkFkZOWkmqsfrjA HTTP/1.1" 200 None


Found playlist: Drum and Bass Top 100 (100 tracks)


In [3]:
spotify_playlist.name

'Drum and Bass Top 100'

Next up we create a corresponding playlist on Tidal (or get it if it already exists):

In [46]:
from plistsync.services.tidal import TidalLibraryCollection, TidalPlaylistCollection

tidal_library = TidalLibraryCollection()

if tidal_library.has_playlist(spotify_playlist.name):
    # Playlist exists, we can use it
    _tidal_playlist = tidal_library.get_playlist(name=spotify_playlist.name)
    assert _tidal_playlist is not None
    tidal_playlist = _tidal_playlist
    print(
        f"Using existing Tidal playlist '{tidal_playlist.id}' "
        f"with {len(tidal_playlist)} tracks"
    )
else:
    # Create the playlist
    resource, lookup = tidal_library.api.playlist.create(
        name=spotify_playlist.name,
        description=spotify_playlist.description or "",
    )
    tidal_playlist = TidalPlaylistCollection.from_response_data(
        tidal_library, resource, lookup
    )
    print(f"Created Tidal playlist '{tidal_playlist.id}'")

INFO:eyconf.config.file:Loading config file: /Users/paul/para/2_Projects/plistsync_25.nosync/config/config.yml
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): openapi.tidal.com:443
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/users/me HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=198373928 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/playlists?filter%5Bowners.id%5D=198373928&page%5Bcursor%5D=eyJpZCI6IjQxZDQxMzNiLWM3YzAtNDY0MS1hM2Y0LWFmODIwNTk3OWYxNiIsIm5hbWUiOiJHb3JpbGxhIE1hbiIsImFkZGVkQXQiOiIyMDI0LTA3LTI1VDE0OjQ2OjE4LjY0M1oiLCJ1cGRhdGVkQXQiOiIyMDI0LTA3LTI1VDE0OjQ2OjIwLjIxMTUyN1oiLCJpdGVtVHlwZSI6IlVTRVJfUExBWUxJU1QiLCJhcnRpc3ROYW1lIjpudWxsLCJyZWxlYXNlZEF0IjpudWxsLCJtaXhUeXBlIjpudWxsfQ&filter%5Bowners.id%5D=198373928 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/users/me HTTP/1.1" 200 None
DEBUG

Using existing Tidal playlist '3202d603-1183-4b57-98a7-ffb24e28ef22' with 72 tracks


We now have both playlists and can start to sync them. We will add tracks that are in the Spotify playlist but not in the Tidal one.

The following might not be super efficient, but it works for demonstration purposes. Maybe we can optimize it later?

In [ ]:
from plistsync.services.spotify import SpotifyPlaylistTrack
from plistsync.services.tidal import TidalPlaylistTrack

# Find tracks in Spotify playlist that are not in Tidal playlist
missing_tracks: list[SpotifyPlaylistTrack] = []

tidal_playlist._refetch_tracks()
print([t for t in tidal_playlist.tracks])

for track_s in spotify_playlist.tracks:
    # Try to match by ISRC only (cutoff 1.0)
    # TODO: implement playlist.match logic (global and local lookup, so match works)
    # matches = tidal_playlist.match(track_s, cutoff=0.8)
    # if len(matches.found) == 0:
    #     missing_tracks.append(track_s)

    if track_s.isrc is not None and track_s.isrc not in [t.isrc for t in tidal_playlist.tracks]:
        missing_tracks.append(track_s)

print(
    f"Found {len(missing_tracks)} missing tracks in Tidal playlist '{tidal_playlist.name}'"
)


tidal_tracks = tidal_library.find_many_by_global_ids(
    map(lambda t: t.global_ids, missing_tracks)
)
tidal_tracks = list(filter(None, tidal_tracks))
print(f"Found {len(tidal_tracks)} out of {len(missing_tracks)} tracks on Tidal")

# service-level tracks have less information than playlist tracks, so we have to convert
tidal_tracks = [TidalPlaylistTrack(t) for t in tidal_tracks]

with tidal_playlist.remote_edit():
    # as of now, inserts are one api call each. (this will take a while)
    tidal_playlist.tracks.extend(tidal_tracks)

print(
    f"Tidal playlist '{tidal_playlist.name}' now has {len(tidal_playlist)} tracks"
)

DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/playlists/3202d603-1183-4b57-98a7-ffb24e28ef22/relationships/items?include=items&include=items.albums&include=items.artists HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/playlists/3202d603-1183-4b57-98a7-ffb24e28ef22/relationships/items?include=items.albums%2Citems.artists&page%5Bcursor%5D=eyJyb3dJZCI6MTU4MjI4MjQ4ODMsImluZGV4Ijo1NzAwMDAwLCJ0aXRsZSI6IkZlZWwgVGhlIFZpYnJhdGlvbiIsImFydGlzdE5hbWUiOiJLYW5pbmUiLCJhbGJ1bVRpdGxlIjoiRmVlbCBUaGUgVmlicmF0aW9uIiwiZGF0ZUFkZGVkIjoiMjAyNi0wMi0xNVQxNTozNDoyOS4yNzkwODEiLCJkdXJhdGlvbiI6MjAyLjB9&include=items&include=items.albums&include=items.artists HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://openapi.tidal.com:443 "GET /v2/playlists/3202d603-1183-4b57-98a7-ffb24e28ef22/relationships/items?include=items.albums%2Citems.artists&page%5Bcursor%5D=eyJyb3dJZCI6MTU4MjI5ODAzOTUsImluZGV4Ijo3NzAwMDAwLCJ0aXRsZSI6IkxldCB0aGUgU3VuIFNoaW5lIiwiYXJ0aX

[Track[Catching Cairo > Generator, hash: -7346083755418667698], Track[Catching Cairo > Sweet Sound, hash: 6170725344074398583], Track[Catching Cairo > Too Fast, hash: 3953572681556161651], Track[Catching Cairo > One Last Tear, hash: 3837009948128667281], Track[Catching Cairo > If Only, hash: -2620328311599840261], Track[Catching Cairo > High Speed Chase, hash: 5249226285896955042], Track[Catching Cairo > Say It Again, hash: -5813203736390777190], Track[Catching Cairo > Tragic, hash: 8653597862485571425], Track[Catching Cairo > Ride The Energy, hash: 7749768645514667291], Track[Catching Cairo > Grindhouse, hash: -1681802516493334326], Track[Catching Cairo > Setting Sun, hash: 597466015894740938], Track[Catching Cairo > Can't Love Me, hash: 7361749255830440964], Track[Catching Cairo > Block Party, hash: -1771931652359850881], Track[Catching Cairo > Dancing In The Leaves, hash: 2536288852649244080], Track[Catching Cairo > Where You Are, hash: -77442447724629094], Track[Catching Cairo > Th

One directional sync is now done. More features coming soon!